# Fitting and transferring discrepancy

A discrepancy describes the part of a measured response that a physical model misses. This tutorial fits a lossy coax with a small conductor error, predicts the correction with its uncertainty, and carries it to a new line length. It then shows how a port discrepancy can carry a joint parameter and correction prediction into a terminated circuit.

The [discrepancy prediction example](../examples/discrepancy_prediction.rst) is the short introduction. The code here uses small synthetic data sets so every result can be checked against a known line.

## A reference line and a shorter target

The reference line is 30 cm long; the target is 15 cm. The nominal line has 10% excess conductor loss. A port correction belongs to the reference response, while an internal correction changes characteristic impedance, attenuation, and phase per unit length. That distinction matters when the line length changes.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pmrf as prf
from pmrf.models import GridLineDiscrepancy, LineCorrected, line_internal_features, reference_line_features
from pmrf.objectives.evaluators import MarginalLogLikelihood
from pmrf.stats.covariance_kernels import RBFKernel
from pmrf.stats.discrepancy_models import GaussianProcess
from pmrf.stats.likelihoods import GaussianLikelihood

tutorial_dir = Path.cwd()
if not (tutorial_dir / "line_internal_discrepancy.py").exists():
    tutorial_dir = Path("docs/tutorials")
sys.path.insert(0, str(tutorial_dir.resolve()))
from line_internal_discrepancy import (
    MODEL_SIGMA, TRUTH_SIGMA, compare_transfer, exact_values,
    frequency, line, s_only_route,
)
from port_discrepancy import run_constrained_transfer

f = frequency()
comparison = compare_transfer()
for name, errors in comparison.items():
    print(f"{name}: RMS S21 error = {errors['rms_s21']:.4g}, eta bias = {errors['eta_bias']:.4g}")

The internal correction follows the shorter line's electrical length. The port correction uses the reference line's S-parameter error as if it belonged to the shorter line.

## Learn an internal correction

Here the reference measurement supplies characteristic impedance $Z_c$ and propagation constant $\gamma$. `reference_line_features` converts them to the four log-ratio channels used by `GridLineDiscrepancy`: real and imaginary $\log Z_c$, log attenuation, and log phase constant. The likelihood variance is specified in this transformed event space.

The nominal model has a free conductor parameter. `predict_joint` returns one Gaussian over that parameter and the correction grid, preserving their covariance.

In [ ]:
reference = line(0.30, TRUTH_SIGMA)
zc, gamma_length = reference.zc_and_gammaL(f)
observed = reference_line_features(zc, gamma_length / 0.30)
nominal = line(0.30, MODEL_SIGMA, free_sigma=True)
reference_likelihood = MarginalLogLikelihood(
    predictor=line_internal_features,
    observed=observed,
    likelihood=GaussianLikelihood(1e-10),
    discrepancy=GaussianProcess(RBFKernel(0.5) * 0.1**2, jitter=1e-12),
)
names, joint = reference_likelihood.predict_joint(nominal, f, f)
print("Joint dimensions:", joint.mean().shape[0], "parameter names:", names)

The same prediction can be attached to a shorter line. The joint prior keeps the conductor parameter correlated with the discrepancy values. `truncate='unnormalised'` respects the conductor's validity range for MAP work; it is unsuitable for sampling or evidence calculations.

In [ ]:
carrier = GridLineDiscrepancy.from_prediction(f, joint, joint=True)
short_line = LineCorrected(line(0.15, MODEL_SIGMA, free_sigma=True), carrier)
short_line = prf.prior(
    short_line, [*names, "discrepancy.values"], joint,
    truncate="unnormalised",
)
truth_short = line(0.15, TRUTH_SIGMA)
rms_s21 = np.sqrt(np.mean(np.abs(np.asarray(short_line.s(f) - truth_short.s(f))[:, 1, 0])**2))
print("Transferred RMS S21 error:", float(rms_s21))

The joint mean contains the conductor parameter first, then four correction channels. Its diagonal gives marginal uncertainty in each channel; the off-diagonal entries still matter when the correction is used with the conductor parameter.

In [ ]:
mean = np.asarray(joint.mean())[1:].reshape(2, 2, f.npoints)
std = np.sqrt(np.diag(np.asarray(joint.covariance()))[1:]).reshape(2, 2, f.npoints)
truth = np.asarray(exact_values(f))
labels = ("log Zc real", "log Zc imaginary", "log attenuation", "log phase")
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for axis, label, predicted, sigma, actual in zip(axes.flat, labels, mean.reshape(4, -1), std.reshape(4, -1), truth.reshape(4, -1)):
    axis.plot(f.f_scaled, actual, label="injected")
    axis.plot(f.f_scaled, predicted, label="predicted")
    axis.fill_between(f.f_scaled, predicted - 2*sigma, predicted + 2*sigma, alpha=0.2)
    axis.set_title(label)
axes[0, 0].legend()
fig.tight_layout()

## When only S-parameters are measured

The preceding route uses $Z_c$ and $\gamma$. If the reference only provides complex S-parameters, a basis representation of the internal correction can be fitted through the line response. The supporting [line transfer script](line_internal_discrepancy.py) carries out the MAP fit, Laplace covariance, and projection onto the target grid. Its four-channel `mean` and `sigma` arrays make the uncertainty visible.

In [ ]:
s_only = s_only_route()
print("Transferred RMS S21 error:", s_only["rms_s21"])
print("Injected values within mean ± 2σ:", s_only["coverage"])

## Port discrepancy in a later circuit

A port correction can instead be transferred when the fitted component itself stays the same and its surrounding circuit changes. The [port transfer script](port_discrepancy.py) fits a coax length and reciprocal port discrepancy, predicts them jointly, and places the corrected cable behind a load. It compares the resulting posterior covariance with a NumPy Kalman update and checks that the transferred prior retains parameter–discrepancy covariance.

In [ ]:
port_transfer = run_constrained_transfer()
print("Reference fit succeeded:", port_transfer["reference_success"])
print("Largest relative covariance-diagonal error:", np.max(port_transfer["relative_diagonal_error"]))
print("Relative prior-precision error:", port_transfer["prior_precision_relative_error"])

The carrier follows what stays physically common between fits: use an internal line correction across lengths, or a port correction when moving the same component into another circuit. In both cases the joint distribution carries parameter–discrepancy covariance into the later calculation.